In [2]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

c:\Users\jaesc2\GitHub\skforecast
0.21.0


In [3]:
# Libraries
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from skforecast.datasets import fetch_dataset
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.plot import plot_prediction_intervals, set_dark_theme

In [4]:
n_rows = 4000
n_exog = 10
steps = 100

data = pd.Series(np.random.rand(n_rows), name='y')
data.index = pd.date_range(start='2000-01-01', periods=n_rows, freq='h')

exog = pd.DataFrame(
    np.random.rand(n_rows, n_exog), columns=[f'exog_{i}' for i in range(n_exog)]
)
exog.index = pd.date_range(start='2000-01-01', periods=n_rows, freq='h')

exog_predict = pd.DataFrame(
    np.random.rand(steps, n_exog), columns=[f'exog_{i}' for i in range(n_exog)]
)
exog_predict.index = pd.date_range(start=data.index[-1] + pd.Timedelta(hours=1), periods=steps, freq='h')

In [5]:
# Create and fit forecaster
# ==============================================================================
forecaster = ForecasterRecursive(
                 estimator       = LGBMRegressor(random_state=123, verbose=-1),
                 lags            = 15,
                 window_features = RollingFeatures(stats=['mean'], window_sizes=10),
                 transformer_y   = None, 
             )

forecaster.fit(y=data, exog=exog, store_in_sample_residuals=True)
forecaster

=================== 
ForecasterRecursive 
=================== 
Estimator: LGBMRegressor 
Lags: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] 
Window features: ['roll_mean_10'] 
Window size: 15 
Series name: y 
Exogenous included: True 
Exogenous names: 
    exog_0, exog_1, exog_2, exog_3, exog_4, exog_5, exog_6, exog_7, exog_8, exog_9 
Transformer for y: None 
Transformer for exog: None 
Weight function included: False 
Differentiation order: None 
Training range: [Timestamp('2000-01-01 00:00:00'), Timestamp('2000-06-15 15:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <Hour> 
Estimator parameters: 
    {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0,
    'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1,
    'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0,
    'n_estimators': 100, 'n_jobs': None, 'num_leaves': 31, 'objective': None,
    'random_state': 123, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0,
    'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': -1} 
fit_kwargs: {} 
Creation date: 2026-03-10 08:26:29 
Last fit date: 2026-03-10 08:26:30 
Skforecast version: 0.21.0 
Python version: 3.13.11 
Forecaster id: None

In [6]:
# Predict
# ==============================================================================
predictions = forecaster.predict(steps=steps, exog=exog_predict)
predictions.head(3)

2000-06-15 16:00:00    0.594802
2000-06-15 17:00:00    0.577321
2000-06-15 18:00:00    0.476855
Freq: h, Name: pred, dtype: float64

## Fit time

In [6]:
%%timeit -n 10 -r 3

forecaster.fit(y=data, exog=exog, store_in_sample_residuals=True)

86 ms ± 9.01 ms per loop (mean ± std. dev. of 3 runs, 10 loops each)


pandas 3

In [7]:
%%timeit -n 10 -r 3

forecaster.fit(y=data, exog=exog, store_in_sample_residuals=True)

107 ms ± 4.63 ms per loop (mean ± std. dev. of 3 runs, 10 loops each)


## Predict time


In [8]:
%%timeit -n 20 -r 5

predictions = forecaster.predict(steps=steps, exog=exog_predict)

17.1 ms ± 481 μs per loop (mean ± std. dev. of 5 runs, 20 loops each)


pandas 3

In [8]:
%%timeit -n 20 -r 5

predictions = forecaster.predict(steps=steps, exog=exog_predict)

23.6 ms ± 3.11 ms per loop (mean ± std. dev. of 5 runs, 20 loops each)
